## Model Export Configuration

The trained lead-scoring pipeline will be stored as a serialized model artifact.

The complete pipeline includes feature engineering, preprocessing, and the trained Gradient Boosting classifier.

Saving the complete pipeline ensures that deployment applications can generate predictions directly from raw lead data without manually repeating the training-time preprocessing steps.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

In [2]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / "lead_scoring_model.pkl"

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Model path:", MODEL_PATH)

Project root: c:\Users\mohsi\Python\Projects\ai-lead-scoring
Models directory: c:\Users\mohsi\Python\Projects\ai-lead-scoring\models
Model path: c:\Users\mohsi\Python\Projects\ai-lead-scoring\models\lead_scoring_model.pkl


## Load and Validate the Exported Model

The exported model is now loaded from disk to verify that it can be used independently of the training notebook.

This validation step is important because the deployment application will load the serialized model artifact rather than using variables from the training environment.

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [5]:
loaded_model = joblib.load(MODEL_PATH)

print("Model loaded successfully!")
print("\nPipeline steps:")
print(loaded_model.named_steps.keys())

Model loaded successfully!

Pipeline steps:
dict_keys(['preprocessor', 'model'])


## Validate Predictions Using Raw Input Data

The exported pipeline is tested using raw lead data from the original dataset.

No manual feature engineering or preprocessing is applied in this notebook.

The loaded pipeline is responsible for applying:

1. Feature engineering
2. Data preprocessing
3. Model prediction

This simulates how the model will be used in the FastAPI deployment environment.

In [6]:
# Define the raw dataset path

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Lead Scoring.csv"

# Load the original raw dataset
df = pd.read_csv(DATA_PATH)

print("Raw dataset loaded successfully!")
print("Dataset shape:", df.shape)

Raw dataset loaded successfully!
Dataset shape: (9240, 37)


In [7]:
# Define the raw input features expected by the model

candidate_features = [
    "Lead Origin",
    "Lead Source",
    "Country",
    "Specialization",
    "How did you hear about X Education",
    "What is your current occupation",
    "What matters most to you in choosing a course",
    "Lead Profile",
    "City",
    "Do Not Email",
    "A free copy of Mastering The Interview",
    "TotalVisits",
    "Total Time Spent on Website",
    "Page Views Per Visit"
]

# Create raw feature matrix
X_raw = df[candidate_features].copy()

print("Raw feature matrix shape:", X_raw.shape)
print("Number of input features:", X_raw.shape[1])

Raw feature matrix shape: (9240, 14)
Number of input features: 14


In [8]:
loaded_model.predict(X_raw)

array([0, 1, 1, ..., 0, 0, 1], shape=(9240,))

In [9]:
loaded_model.predict_proba(X_raw)

array([[0.64648104, 0.35351896],
       [0.43115761, 0.56884239],
       [0.14182927, 0.85817073],
       ...,
       [0.81087298, 0.18912702],
       [0.90017947, 0.09982053],
       [0.12210445, 0.87789555]], shape=(9240, 2))

## End-to-End Prediction Validation

The exported model is tested using raw lead data without manually applying feature engineering or preprocessing.

Successful predictions confirm that the saved pipeline contains all required transformations and can be used directly in the deployment environment.

In [10]:
# Generate predictions from raw data

predictions = loaded_model.predict(X_raw)
probabilities = loaded_model.predict_proba(X_raw)[:, 1]

print("End-to-end prediction validation successful!")

print("\nNumber of predictions:", len(predictions))
print("Prediction shape:", predictions.shape)
print("Probability shape:", probabilities.shape)

print("\nFirst 5 predictions:")
print(predictions[:5])

print("\nFirst 5 conversion probabilities:")
print(probabilities[:5])

End-to-end prediction validation successful!

Number of predictions: 9240
Prediction shape: (9240,)
Probability shape: (9240,)

First 5 predictions:
[0 1 1 0 1]

First 5 conversion probabilities:
[0.35351896 0.56884239 0.85817073 0.12883588 0.61443258]


## Single Lead Prediction Validation

The exported model is tested using a single manually created lead record.

This simulates a real deployment scenario where new lead information is received through an API or user interface.

The model should generate a prediction and conversion probability directly from raw input data.

In [11]:
# Create a single sample lead using raw model input features

sample_lead = pd.DataFrame([{
    "Lead Origin": "Landing Page Submission",
    "Lead Source": "Google",
    "Country": "India",
    "Specialization": "Business Administration",
    "How did you hear about X Education": "Online Search",
    "What is your current occupation": "Working Professional",
    "What matters most to you in choosing a course": "Better Career Prospects",
    "Lead Profile": "Potential Lead",
    "City": "Mumbai",
    "Do Not Email": "No",
    "A free copy of Mastering The Interview": "No",
    "TotalVisits": 5,
    "Total Time Spent on Website": 1200,
    "Page Views Per Visit": 4
}])

print("Sample lead created successfully!")
print("\nInput shape:", sample_lead.shape)

sample_lead

Sample lead created successfully!

Input shape: (1, 14)


,Lead Origin,Lead Source,Country,Specialization,How did you hear about X Education,What is your current occupation,What matters most to you in choosing a course,Lead Profile,City,Do Not Email,A free copy of Mastering The Interview,TotalVisits,Total Time Spent on Website,Page Views Per Visit
0,Landing Page Submission,Google,India,Business Administration,Online Search,Working Professional,Better Career Prospects,Potential Lead,Mumbai,No,No,5,1200,4


In [12]:
# Generate prediction and conversion probability

prediction = loaded_model.predict(sample_lead)[0]

conversion_probability = loaded_model.predict_proba(
    sample_lead
)[0, 1]

print("Prediction:", prediction)
print(
    "Conversion Probability:",
    f"{conversion_probability:.2%}"
)

Prediction: 1
Conversion Probability: 97.07%


## Lead Priority Assignment

Conversion probability is translated into a business-oriented priority level to help sales teams focus their efforts.

The priority strategy is based on the threshold analysis performed during model training:

- High Priority: probability greater than or equal to 70%
- Medium Priority: probability between 40% and 70%
- Low Priority: probability below 40%

In [13]:
# Assign a business priority based on conversion probability

def assign_lead_priority(probability):
    if probability >= 0.70:
        return "High"
    elif probability >= 0.40:
        return "Medium"
    else:
        return "Low"


lead_priority = assign_lead_priority(conversion_probability)

print("Prediction:", prediction)
print(f"Conversion Probability: {conversion_probability:.2%}")
print("Lead Priority:", lead_priority)

Prediction: 1
Conversion Probability: 97.07%
Lead Priority: High


In [14]:
{
    "prediction": "Likely to Convert",
    "conversion_probability": 0.9707,
    "lead_priority": "High"
}

{'prediction': 'Likely to Convert',
 'conversion_probability': 0.9707,
 'lead_priority': 'High'}


## Model Export Validation Summary

The exported lead-scoring pipeline was successfully validated outside the training notebook.

The validation confirmed that the model can:

- Load successfully from the exported model file
- Accept raw lead data without manual preprocessing
- Generate conversion predictions
- Calculate conversion probabilities
- Score a single new lead
- Assign a business-oriented lead priority

The exported model is ready to be integrated into the FastAPI backend.